In [4]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.patches import FancyBboxPatch
from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import os
import glob
import json
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.titleweight'] = 'bold'

class ContactMapLoader:
    def __init__(self, npy_dir, pattern="*.npz", verbose=True):
        self.npy_dir = npy_dir
        self.pattern = pattern
        self.verbose = verbose
        self.contact_maps = None
        self.filenames = None
        self.metadata = {}
        
    def load_maps(self):
        npy_files = sorted(glob.glob(os.path.join(self.npy_dir, "*.npy")))
        npz_files = sorted(glob.glob(os.path.join(self.npy_dir, "*.npz")))
        
        if npz_files:
            all_files = npz_files
            file_type = 'npz'
        elif npy_files:
            all_files = npy_files
            file_type = 'npy'
        else:
            raise ValueError(f"No .npy or .npz files found in {self.npy_dir}")
        
        contact_maps = []
        filenames = []
        
        if self.verbose:
            print(f"Loading contact maps from: {self.npy_dir}")
            print(f"Found {len(all_files)} {file_type} files")
        
        for i, file_path in enumerate(all_files):
            try:
                if file_type == 'npz':
                    data = np.load(file_path)
                    contact_map = data[data.files[0]]
                else:
                    contact_map = np.load(file_path)
                
                contact_maps.append(contact_map)
                filenames.append(Path(file_path).name)
                
                if self.verbose and (i + 1) % 10 == 0:
                    print(f"  Loaded {i + 1}/{len(all_files)} files...")
                    
            except Exception as e:
                print(f"Warning: Failed to load {file_path}: {str(e)}")
                continue
        
        self.contact_maps = np.array(contact_maps)
        self.filenames = filenames
        
        self.metadata = {
            'n_maps': len(self.contact_maps),
            'map_shape': self.contact_maps[0].shape,
            'n_residues': self.contact_maps[0].shape[0],
            'load_time': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'file_type': file_type
        }
        
        if self.verbose:
            print(f"\n✓ Successfully loaded {len(self.contact_maps)} contact maps")
            print(f"  Contact map shape: {self.contact_maps[0].shape}")
            print(f"  Matrix dimensions: {self.contact_maps[0].shape[0]} x {self.contact_maps[0].shape[1]}")
            print(f"  File type: {file_type}")
        
        return self.contact_maps, self.filenames
    
    def validate_maps(self):
        if self.contact_maps is None:
            raise ValueError("No contact maps loaded. Call load_maps() first.")
        
        issues = []
        is_rectangular = False
        
        if np.isnan(self.contact_maps).any():
            issues.append("NaN values detected")
        
        if (self.contact_maps < 0).any():
            issues.append("Negative values detected")
        
        for i, cm in enumerate(self.contact_maps):
            if cm.shape[0] != cm.shape[1]:
                is_rectangular = True
                if self.verbose and i == 0:
                    print(f"ℹ  Note: Contact maps are rectangular ({cm.shape[0]}x{cm.shape[1]})")
                    print(f"   This represents inter-domain contacts (e.g., DNA-protein)")
                    print(f"   Skipping symmetry check (not applicable for rectangular matrices)")
                break
            else:
                if not np.allclose(cm, cm.T, rtol=1e-5):
                    issues.append(f"Map {i} is not symmetric")
                    break
        
        if issues:
            print("⚠  Validation warnings:")
            for issue in issues:
                print(f"   - {issue}")
        elif not is_rectangular:
            if self.verbose:
                print("✓ All contact maps validated successfully")
        
        return len(issues) == 0

class ContactFeatureExtractor:
    def __init__(self, contact_maps, verbose=True):
        self.contact_maps = contact_maps
        self.verbose = verbose
        self.features = None
        self.feature_info = {}
    
    def extract_upper_triangle(self, exclude_diagonal=True):
        n_windows = self.contact_maps.shape[0]
        
        if self.contact_maps[0].shape[0] != self.contact_maps[0].shape[1]:
            if self.verbose:
                print(f"ℹ  Matrix is rectangular - extracting all elements (flattened)")
            features = self.contact_maps.reshape(n_windows, -1)
            self.feature_info = {
                'method': 'rectangular_flatten',
                'n_features': features.shape[1],
                'shape': features.shape
            }
        else:
            if exclude_diagonal:
                triu_indices = np.triu_indices_from(self.contact_maps[0], k=1)
            else:
                triu_indices = np.triu_indices_from(self.contact_maps[0], k=0)
            
            features = np.array([cm[triu_indices] for cm in self.contact_maps])
            self.feature_info = {
                'method': 'upper_triangle',
                'exclude_diagonal': exclude_diagonal,
                'n_features': features.shape[1],
                'shape': features.shape
            }
        
        self.features = features
        
        if self.verbose:
            print(f"\nExtracted features:")
            print(f"  Feature matrix shape: {features.shape}")
            print(f"  Number of features per window: {features.shape[1]}")
        
        return features
    
    def extract_domain_features(self, dna_residues, van_residues, combine=True):
        dna_idx = np.array(dna_residues)
        van_idx = np.array(van_residues)
        
        features_dict = {}
        
        dna_dna_contacts = self.contact_maps[:, dna_idx[:, None], dna_idx]
        triu_dna = np.triu_indices(len(dna_idx), k=1)
        features_dict['dna_dna'] = np.array([cm[triu_dna] for cm in dna_dna_contacts])
        
        van_van_contacts = self.contact_maps[:, van_idx[:, None], van_idx]
        triu_van = np.triu_indices(len(van_idx), k=1)
        features_dict['van_van'] = np.array([cm[triu_van] for cm in van_van_contacts])
        
        dna_van_contacts = self.contact_maps[:, dna_idx[:, None], van_idx]
        features_dict['dna_van'] = dna_van_contacts.reshape(len(self.contact_maps), -1)
        
        if self.verbose:
            print(f"\nExtracted domain-specific features:")
            for key, val in features_dict.items():
                print(f"  {key:12s}: {val.shape}")
        
        if combine:
            features_dict['combined'] = np.hstack([
                features_dict['dna_dna'],
                features_dict['van_van'],
                features_dict['dna_van']
            ])
            
            self.features = features_dict['combined']
            self.feature_info = {
                'method': 'domain_features',
                'dna_residues': len(dna_idx),
                'van_residues': len(van_idx),
                'n_features': features_dict['combined'].shape[1],
                'shape': features_dict['combined'].shape
            }
            
            if self.verbose:
                print(f"  {'combined':12s}: {features_dict['combined'].shape}")
            
            return features_dict['combined']
        else:
            return features_dict

class DNAFoldPCA:
    def __init__(self, features, standardize=True, verbose=True):
        self.features = features
        self.standardize = standardize
        self.verbose = verbose
        
        self.pca = None
        self.scaler = None
        self.transformed = None
        self.results = {}
    
    def preprocess_features(self):
        features = self.features.copy()
        
        if np.isnan(features).any():
            if self.verbose:
                print("⚠ Warning: NaN values found. Replacing with 0.")
            features = np.nan_to_num(features, nan=0.0)
        
        if np.isinf(features).any():
            if self.verbose:
                print("⚠ Warning: Infinite values found. Replacing with 0.")
            features = np.nan_to_num(features, posinf=0.0, neginf=0.0)
        
        if self.standardize:
            self.scaler = StandardScaler()
            features = self.scaler.fit_transform(features)
            if self.verbose:
                print("✓ Features standardized (zero mean, unit variance)")
        
        return features
    
    def fit(self, n_components=10):
        if self.verbose:
            print(f"\n{'='*70}")
            print("PERFORMING PCA ANALYSIS")
            print(f"{'='*70}")
        
        features_scaled = self.preprocess_features()
        
        self.pca = PCA(n_components=n_components)
        self.transformed = self.pca.fit_transform(features_scaled)
        
        self.results = {
            'n_components': len(self.pca.explained_variance_ratio_),
            'explained_variance': self.pca.explained_variance_,
            'explained_variance_ratio': self.pca.explained_variance_ratio_,
            'cumulative_variance': np.cumsum(self.pca.explained_variance_ratio_),
            'components': self.pca.components_,
            'mean': self.pca.mean_,
            'transformed': self.transformed
        }
        
        if self.verbose:
            self._print_results()
        
        return self
    
    def _print_results(self):
        print(f"\nPCA Results Summary:")
        print(f"  Number of components: {self.results['n_components']}")
        print(f"  Total variance explained: {self.results['cumulative_variance'][-1]:.4f}")
        
        print(f"\n  Variance explained by each PC:")
        for i in range(min(10, self.results['n_components'])):
            var_ratio = self.results['explained_variance_ratio'][i]
            cum_var = self.results['cumulative_variance'][i]
            print(f"    PC{i+1:2d}: {var_ratio:7.4f} ({var_ratio*100:5.2f}%)  |  "
                  f"Cumulative: {cum_var:.4f} ({cum_var*100:5.2f}%)")
        
        if self.results['n_components'] > 10:
            print(f"    ... ({self.results['n_components'] - 10} more components)")

class FoldingDynamicsAnalyzer:
    def __init__(self, transformed, verbose=True):
        self.transformed = transformed
        self.verbose = verbose
        self.metrics = {}
    
    def calculate_trajectory_metrics(self):
        if self.verbose:
            print(f"\n{'='*70}")
            print("ANALYZING FOLDING DYNAMICS")
            print(f"{'='*70}")
        
        pc_trajectory_length = np.sum(
            np.linalg.norm(np.diff(self.transformed[:, :3], axis=0), axis=1)
        )
        
        distances_from_initial = np.linalg.norm(
            self.transformed - self.transformed[0], axis=1
        )
        
        distances_from_final = np.linalg.norm(
            self.transformed - self.transformed[-1], axis=1
        )
        
        mean_structure = self.transformed.mean(axis=0)
        rmsd_pc = np.sqrt(np.mean((self.transformed - mean_structure)**2, axis=1))
        
        rog_pc = np.sqrt(np.mean(np.sum((self.transformed - mean_structure)**2, axis=1)))
        
        self.metrics = {
            'trajectory_length': pc_trajectory_length,
            'distances_from_initial': distances_from_initial,
            'distances_from_final': distances_from_final,
            'rmsd_pc': rmsd_pc,
            'radius_of_gyration': rog_pc,
            'mean_structure': mean_structure
        }
        
        if self.verbose:
            print(f"\nTrajectory Metrics:")
            print(f"  Total trajectory length (PC1-3): {pc_trajectory_length:.4f}")
            print(f"  Max distance from initial: {distances_from_initial.max():.4f}")
            print(f"  Max distance from final: {distances_from_final.max():.4f}")
            print(f"  Mean RMSD in PC space: {rmsd_pc.mean():.4f}")
            print(f"  Radius of gyration: {rog_pc:.4f}")
        
        return self.metrics
    
    def identify_folding_states(self, n_clusters=3):
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        labels = kmeans.fit_predict(self.transformed[:, :3])
        
        self.metrics['folding_states'] = labels
        self.metrics['n_clusters'] = n_clusters
        self.metrics['cluster_centers'] = kmeans.cluster_centers_
        
        if self.verbose:
            print(f"\nIdentified {n_clusters} folding states:")
            for i in range(n_clusters):
                count = np.sum(labels == i)
                percentage = count / len(labels) * 100
                print(f"  State {i+1}: {count:4d} windows ({percentage:5.1f}%)")
        
        return labels

class PCAVisualizer:
    def __init__(self, pca_results, output_dir="pca_output"):
        self.pca_results = pca_results
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
    
    def plot_complete_trajectory(self, folding_states=None, save=True):
        fig, ax = plt.subplots(figsize=(14, 11))
        
        transformed = self.pca_results['transformed']
        var1 = self.pca_results['explained_variance_ratio'][0] * 100
        var2 = self.pca_results['explained_variance_ratio'][1] * 100
        
        n_points = len(transformed)
        time_colors = plt.cm.plasma(np.linspace(0, 1, n_points))
        
        if folding_states is not None:
            unique_states = np.unique(folding_states)
            state_colors = plt.cm.Set3(np.linspace(0, 1, len(unique_states)))
            
            for idx, state in enumerate(unique_states):
                mask = folding_states == state
                ax.scatter(transformed[mask, 0], transformed[mask, 1],
                          c=[state_colors[idx]], s=180, alpha=0.7,
                          edgecolors='black', linewidth=1.2,
                          label=f'State {state+1}', zorder=5)
        else:
            scatter = ax.scatter(transformed[:, 0], transformed[:, 1],
                               c=np.arange(n_points), cmap='plasma', s=180,
                               alpha=0.7, edgecolors='black', linewidth=1.2,
                               zorder=5)
            cbar = plt.colorbar(scatter, ax=ax, pad=0.02)
            cbar.set_label('Time Window Index', fontsize=13, fontweight='bold')
            cbar.ax.tick_params(labelsize=11)
        
        ax.plot(transformed[:, 0], transformed[:, 1],
               'k-', alpha=0.35, linewidth=2, zorder=3, label='Trajectory Path')
        
        ax.scatter(transformed[0, 0], transformed[0, 1],
                  c='lime', s=500, marker='*', edgecolors='darkgreen',
                  linewidths=3, label='Initial Structure (t=0)', zorder=10)
        
        ax.scatter(transformed[-1, 0], transformed[-1, 1],
                  c='red', s=500, marker='*', edgecolors='darkred',
                  linewidths=3, label='Final Structure (t=end)', zorder=10)
        
        n_arrows = min(5, n_points // 10)
        if n_arrows > 0:
            arrow_indices = np.linspace(0, n_points-2, n_arrows, dtype=int)
            for idx in arrow_indices:
                dx = transformed[idx+1, 0] - transformed[idx, 0]
                dy = transformed[idx+1, 1] - transformed[idx, 1]
                ax.arrow(transformed[idx, 0], transformed[idx, 1], dx, dy,
                        head_width=0.3, head_length=0.2, fc='gray', ec='gray',
                        alpha=0.4, linewidth=1.5, zorder=2)
        
        ax.set_xlabel(f'Principal Component 1 ({var1:.2f}% variance explained)',
                     fontsize=14, fontweight='bold')
        ax.set_ylabel(f'Principal Component 2 ({var2:.2f}% variance explained)',
                     fontsize=14, fontweight='bold')
        ax.set_title('DNA Folding Trajectory in Principal Component Space\n' +
                    'Color-coded by time progression with folding state annotations',
                    fontsize=16, fontweight='bold', pad=20)
        
        legend = ax.legend(fontsize=12, loc='best', framealpha=0.95,
                          edgecolor='black', fancybox=True, shadow=True)
        legend.set_zorder(20)
        
        ax.grid(True, alpha=0.4, linestyle='--', linewidth=0.8)
        ax.tick_params(labelsize=12)
        
        ax.text(0.02, 0.98, f'Total Windows: {n_points}\nTrajectory Length: {np.sum(np.linalg.norm(np.diff(transformed[:,:2], axis=0), axis=1)):.2f}',
                transform=ax.transAxes, fontsize=11, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
        
        plt.tight_layout()
        
        if save:
            plt.savefig(os.path.join(self.output_dir, 'Figure1_Complete_Folding_Trajectory.png'),
                       dpi=300, bbox_inches='tight', facecolor='white')
            print(f"✓ Saved: Figure1_Complete_Folding_Trajectory.png")
        
        plt.close()
    
    def plot_pc_evolution_multipanel(self, folding_states=None, n_pcs=5, save=True):
        transformed = self.pca_results['transformed']
        n_pcs = min(n_pcs, transformed.shape[1])
        
        fig, axes = plt.subplots(n_pcs, 1, figsize=(16, 3.5*n_pcs))
        
        if n_pcs == 1:
            axes = [axes]
        
        time_points = np.arange(len(transformed))
        
        colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
        
        for i in range(n_pcs):
            var = self.pca_results['explained_variance_ratio'][i] * 100
            
            if folding_states is not None:
                unique_states = np.unique(folding_states)
                for state in unique_states:
                    mask = folding_states == state
                    axes[i].plot(time_points[mask], transformed[mask, i],
                               'o', markersize=7, alpha=0.7,
                               label=f'State {state+1}')
            
            axes[i].plot(time_points, transformed[:, i],
                        '-', linewidth=2.5, color=colors[i % len(colors)],
                        alpha=0.6, label=f'PC{i+1} Trajectory')
            
            axes[i].axhline(y=0, color='black', linestyle='--',
                          linewidth=1.5, alpha=0.5, label='Zero Reference')
            
            axes[i].set_ylabel(f'PC{i+1} Score\n({var:.2f}% var)',
                             fontsize=13, fontweight='bold')
            axes[i].set_title(f'Principal Component {i+1} Evolution Over Time',
                            fontsize=14, fontweight='bold', pad=10)
            axes[i].grid(True, alpha=0.4, linestyle='--', linewidth=0.8)
            axes[i].tick_params(labelsize=11)
            axes[i].legend(fontsize=10, loc='best', framealpha=0.9)
            
            y_range = transformed[:, i].max() - transformed[:, i].min()
            if y_range > 0:
                axes[i].set_ylim(transformed[:, i].min() - 0.1*y_range,
                               transformed[:, i].max() + 0.1*y_range)
        
        axes[-1].set_xlabel('Time Window Index', fontsize=14, fontweight='bold')
        
        fig.suptitle('Multi-Panel PC Evolution Dashboard\n' +
                    'Temporal dynamics of principal components across folding trajectory',
                    fontsize=16, fontweight='bold', y=0.995)
        
        plt.tight_layout(rect=[0, 0, 1, 0.99])
        
        if save:
            plt.savefig(os.path.join(self.output_dir, 'Figure2_PC_Evolution_Multipanel.png'),
                       dpi=300, bbox_inches='tight', facecolor='white')
            print(f"✓ Saved: Figure2_PC_Evolution_Multipanel.png")
        
        plt.close()
    
    def plot_folding_metrics_dashboard(self, metrics, folding_states=None, save=True):
        fig, axes = plt.subplots(2, 2, figsize=(18, 14))
        
        time_points = np.arange(len(metrics['distances_from_initial']))
        
        axes[0, 0].plot(time_points, metrics['distances_from_initial'],
                       'o-', linewidth=3, markersize=6, color='#1f77b4',
                       label='Distance from Initial')
        axes[0, 0].fill_between(time_points, 0, metrics['distances_from_initial'],
                               alpha=0.2, color='#1f77b4')
        axes[0, 0].set_xlabel('Time Window Index', fontsize=13, fontweight='bold')
        axes[0, 0].set_ylabel('Euclidean Distance in PC Space', fontsize=13, fontweight='bold')
        axes[0, 0].set_title('Distance from Initial Structure\n(Folding Progression Metric)',
                           fontsize=14, fontweight='bold', pad=15)
        axes[0, 0].grid(True, alpha=0.4, linestyle='--', linewidth=0.8)
        axes[0, 0].tick_params(labelsize=11)
        axes[0, 0].legend(fontsize=11, loc='best', framealpha=0.9)
        
        axes[0, 1].plot(time_points, metrics['distances_from_final'],
                       'o-', linewidth=3, markersize=6, color='#ff7f0e',
                       label='Distance from Final')
        axes[0, 1].fill_between(time_points, 0, metrics['distances_from_final'],
                               alpha=0.2, color='#ff7f0e')
        axes[0, 1].set_xlabel('Time Window Index', fontsize=13, fontweight='bold')
        axes[0, 1].set_ylabel('Euclidean Distance in PC Space', fontsize=13, fontweight='bold')
        axes[0, 1].set_title('Distance from Final Structure\n(Convergence to Target)',
                           fontsize=14, fontweight='bold', pad=15)
        axes[0, 1].grid(True, alpha=0.4, linestyle='--', linewidth=0.8)
        axes[0, 1].tick_params(labelsize=11)
        axes[0, 1].legend(fontsize=11, loc='best', framealpha=0.9)
        
        axes[1, 0].plot(time_points, metrics['rmsd_pc'],
                       'o-', linewidth=3, markersize=6, color='#2ca02c',
                       label='RMSD in PC Space')
        axes[1, 0].fill_between(time_points, metrics['rmsd_pc'].min(),
                               metrics['rmsd_pc'], alpha=0.2, color='#2ca02c')
        axes[1, 0].axhline(y=metrics['rmsd_pc'].mean(), color='red',
                          linestyle='--', linewidth=2, alpha=0.7,
                          label=f'Mean RMSD: {metrics["rmsd_pc"].mean():.2f}')
        axes[1, 0].set_xlabel('Time Window Index', fontsize=13, fontweight='bold')
        axes[1, 0].set_ylabel('RMSD Value', fontsize=13, fontweight='bold')
        axes[1, 0].set_title('Root Mean Square Deviation in PC Space\n(Structural Fluctuation)',
                           fontsize=14, fontweight='bold', pad=15)
        axes[1, 0].grid(True, alpha=0.4, linestyle='--', linewidth=0.8)
        axes[1, 0].tick_params(labelsize=11)
        axes[1, 0].legend(fontsize=11, loc='best', framealpha=0.9)
        
        axes[1, 1].plot(time_points, metrics['distances_from_initial'],
                       '-', linewidth=3.5, color='#1f77b4', alpha=0.8,
                       label='From Initial Structure')
        axes[1, 1].plot(time_points, metrics['distances_from_final'],
                       '-', linewidth=3.5, color='#ff7f0e', alpha=0.8,
                       label='From Final Structure')
        
        crossover = None
        for i in range(len(time_points)-1):
            if (metrics['distances_from_initial'][i] <= metrics['distances_from_final'][i] and
                metrics['distances_from_initial'][i+1] > metrics['distances_from_final'][i+1]):
                crossover = i
                break
        
        if crossover is not None:
            axes[1, 1].axvline(x=crossover, color='green', linestyle='--',
                             linewidth=2.5, alpha=0.7,
                             label=f'Transition Point (t={crossover})')
            axes[1, 1].scatter(crossover, metrics['distances_from_initial'][crossover],
                             s=200, c='green', marker='o', edgecolors='darkgreen',
                             linewidths=2, zorder=10)
        
        axes[1, 1].set_xlabel('Time Window Index', fontsize=13, fontweight='bold')
        axes[1, 1].set_ylabel('Euclidean Distance in PC Space', fontsize=13, fontweight='bold')
        axes[1, 1].set_title('Combined Folding Progression Analysis\n(Initial vs Final Distance Comparison)',
                           fontsize=14, fontweight='bold', pad=15)
        axes[1, 1].grid(True, alpha=0.4, linestyle='--', linewidth=0.8)
        axes[1, 1].tick_params(labelsize=11)
        axes[1, 1].legend(fontsize=11, loc='best', framealpha=0.9)
        
        fig.suptitle('DNA Folding Dynamics Metrics Dashboard\n' +
                    'Comprehensive analysis of structural evolution in principal component space',
                    fontsize=16, fontweight='bold', y=0.995)
        
        plt.tight_layout(rect=[0, 0, 1, 0.99])
        
        if save:
            plt.savefig(os.path.join(self.output_dir, 'Figure3_Folding_Metrics_Dashboard.png'),
                       dpi=300, bbox_inches='tight', facecolor='white')
            print(f"✓ Saved: Figure3_Folding_Metrics_Dashboard.png")
        
        plt.close()
    
    def plot_variance_analysis(self, save=True):
        fig, ax = plt.subplots(figsize=(14, 8))
        
        n_components = len(self.pca_results['explained_variance_ratio'])
        x = np.arange(1, n_components + 1)
        
        bars = ax.bar(x, self.pca_results['explained_variance_ratio'],
                     alpha=0.75, color='steelblue', edgecolor='black',
                     linewidth=1.5, label='Individual Variance')
        
        for i, bar in enumerate(bars):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                   f'{height*100:.1f}%', ha='center', va='bottom',
                   fontsize=10, fontweight='bold')
        
        ax.plot(x, self.pca_results['cumulative_variance'],
               'ro-', linewidth=3, markersize=10, markeredgecolor='darkred',
               markeredgewidth=2, label='Cumulative Variance', zorder=10)
        
        ax.axhline(y=0.8, color='green', linestyle='--', linewidth=2,
                  alpha=0.7, label='80% Variance Threshold')
        ax.axhline(y=0.9, color='orange', linestyle='--', linewidth=2,
                  alpha=0.7, label='90% Variance Threshold')
        
        cumsum = self.pca_results['cumulative_variance']
        pc_80 = np.argmax(cumsum >= 0.8) + 1 if np.any(cumsum >= 0.8) else n_components
        pc_90 = np.argmax(cumsum >= 0.9) + 1 if np.any(cumsum >= 0.9) else n_components
        
        if pc_80 <= n_components:
            ax.axvline(x=pc_80, color='green', linestyle=':', linewidth=2, alpha=0.5)
            ax.text(pc_80, 0.82, f'PC{pc_80}', fontsize=11, fontweight='bold',
                   bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))
        
        if pc_90 <= n_components:
            ax.axvline(x=pc_90, color='orange', linestyle=':', linewidth=2, alpha=0.5)
            ax.text(pc_90, 0.92, f'PC{pc_90}', fontsize=11, fontweight='bold',
                   bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
        
        ax.set_xlabel('Principal Component Number', fontsize=14, fontweight='bold')
        ax.set_ylabel('Proportion of Variance Explained', fontsize=14, fontweight='bold')
        ax.set_title('PCA Variance Analysis: Individual and Cumulative Contributions\n' +
                    'Optimal component selection based on explained variance thresholds',
                    fontsize=16, fontweight='bold', pad=20)
        ax.legend(fontsize=12, loc='center right', framealpha=0.95,
                 edgecolor='black', fancybox=True)
        ax.grid(True, alpha=0.4, linestyle='--', linewidth=0.8)
        ax.set_xlim(0.5, n_components + 0.5)
        ax.set_ylim(0, 1.05)
        ax.tick_params(labelsize=12)
        
        plt.tight_layout()
        
        if save:
            plt.savefig(os.path.join(self.output_dir, 'Figure4_Variance_Analysis.png'),
                       dpi=300, bbox_inches='tight', facecolor='white')
            print(f"✓ Saved: Figure4_Variance_Analysis.png")
        
        plt.close()
    
    def plot_all(self, folding_states=None, metrics=None):
        print(f"\n{'='*70}")
        print("GENERATING HIGH-QUALITY VISUALIZATIONS")
        print(f"{'='*70}\n")
        
        self.plot_complete_trajectory(folding_states)
        self.plot_pc_evolution_multipanel(folding_states)
        
        if metrics is not None:
            self.plot_folding_metrics_dashboard(metrics, folding_states)
        
        self.plot_variance_analysis()
        
        print(f"\n✓ All publication-quality plots saved to: {self.output_dir}/")

class DNAFoldPCAPipeline:
    def __init__(self, npy_dir, output_dir="pca_output", verbose=True):
        self.npy_dir = npy_dir
        self.output_dir = output_dir
        self.verbose = verbose
        
        os.makedirs(output_dir, exist_ok=True)
        
        self.loader = None
        self.extractor = None
        self.pca_analyzer = None
        self.dynamics_analyzer = None
        self.visualizer = None
        
        self.results = {}
    
    def run(self, n_components=10, n_clusters=3,
            dna_residues=None, van_residues=None,
            feature_method='upper_triangle'):
        
        print(f"\n{'='*70}")
        print("DNA FOLD AND CONTACT MAP PCA ANALYSIS PIPELINE")
        print(f"{'='*70}")
        print(f"Input directory: {self.npy_dir}")
        print(f"Output directory: {self.output_dir}")
        print(f"{'='*70}\n")
        
        print("[Step 1/6] Loading contact maps...")
        self.loader = ContactMapLoader(self.npy_dir, verbose=self.verbose)
        contact_maps, filenames = self.loader.load_maps()
        self.loader.validate_maps()
        
        print(f"\n[Step 2/6] Extracting features (method: {feature_method})...")
        self.extractor = ContactFeatureExtractor(contact_maps, verbose=self.verbose)
        
        if feature_method == 'domain' and dna_residues and van_residues:
            features = self.extractor.extract_domain_features(dna_residues, van_residues)
        else:
            features = self.extractor.extract_upper_triangle()
        
        print(f"\n[Step 3/6] Performing PCA analysis...")
        self.pca_analyzer = DNAFoldPCA(features, verbose=self.verbose)
        self.pca_analyzer.fit(n_components=n_components)
        
        print(f"\n[Step 4/6] Analyzing folding dynamics...")
        self.dynamics_analyzer = FoldingDynamicsAnalyzer(
            self.pca_analyzer.transformed, verbose=self.verbose
        )
        metrics = self.dynamics_analyzer.calculate_trajectory_metrics()
        folding_states = self.dynamics_analyzer.identify_folding_states(n_clusters)
        
        print(f"\n[Step 5/6] Creating publication-quality visualizations...")
        self.visualizer = PCAVisualizer(
            self.pca_analyzer.results, output_dir=self.output_dir
        )
        self.visualizer.plot_all(folding_states=folding_states, metrics=metrics)
        
        print(f"\n[Step 6/6] Saving results...")
        self._save_results(metrics, folding_states, filenames)
        
        self.results = {
            'contact_maps': contact_maps,
            'filenames': filenames,
            'features': features,
            'pca': self.pca_analyzer.pca,
            'transformed': self.pca_analyzer.transformed,
            'scaler': self.pca_analyzer.scaler,
            'pca_results': self.pca_analyzer.results,
            'metrics': metrics,
            'folding_states': folding_states,
            'metadata': self.loader.metadata
        }
        
        print(f"\n{'='*70}")
        print("✓ ANALYSIS COMPLETE!")
        print(f"{'='*70}")
        print(f"Results saved to: {self.output_dir}/")
        print(f"{'='*70}\n")
        
        return self.results
    
    def _save_results(self, metrics, folding_states, filenames):
        np.save(os.path.join(self.output_dir, 'pca_transformed.npy'),
                self.pca_analyzer.transformed)
        np.save(os.path.join(self.output_dir, 'folding_states.npy'),
                folding_states)
        np.save(os.path.join(self.output_dir, 'explained_variance.npy'),
                self.pca_analyzer.results['explained_variance_ratio'])
        
        metadata = {
            'analysis_date': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'n_contact_maps': len(filenames),
            'n_components': self.pca_analyzer.results['n_components'],
            'total_variance_explained': float(
                self.pca_analyzer.results['cumulative_variance'][-1]
            ),
            'n_folding_states': len(np.unique(folding_states)),
            'trajectory_length': float(metrics['trajectory_length']),
            'filenames': filenames
        }
        
        with open(os.path.join(self.output_dir, 'analysis_metadata.json'), 'w') as f:
            json.dump(metadata, f, indent=4)
        
        print(f"\n✓ Saved numerical results:")
        print(f"  - pca_transformed.npy")
        print(f"  - folding_states.npy")
        print(f"  - explained_variance.npy")
        print(f"  - analysis_metadata.json")

NPY_DIR = "/kaggle/input/cross-validation-dataset-2/give_partham_data/contactmap/system1/npy_data"
OUTPUT_DIR = "/kaggle/working/PCA_System - 2"
N_COMPONENTS = 10
N_CLUSTERS = 3
FEATURE_METHOD = 'upper_triangle'
DNA_RESIDUES = None
VAN_RESIDUES = None

pipeline = DNAFoldPCAPipeline(
    npy_dir=NPY_DIR,
    output_dir=OUTPUT_DIR,
    verbose=True
)

results = pipeline.run(
    n_components=N_COMPONENTS,
    n_clusters=N_CLUSTERS,
    dna_residues=DNA_RESIDUES,
    van_residues=VAN_RESIDUES,
    feature_method=FEATURE_METHOD
)

print("\n" + "="*70)
print("ANALYSIS RESULTS SUMMARY")
print("="*70)
print(f"Number of contact maps analyzed: {len(results['filenames'])}")
print(f"Number of PCs computed: {results['pca_results']['n_components']}")
print(f"Total variance explained: {results['pca_results']['cumulative_variance'][-1]:.4f}")
print(f"Number of folding states identified: {len(np.unique(results['folding_states']))}")
print("="*70)

from IPython.display import Image, display

print("\nDisplaying Generated Figures:")
print("="*70)

display(Image('/kaggle/working/PCA_System - 2/Figure1_Complete_Folding_Trajectory.png'))
display(Image('/kaggle/working/PCA_System - 2/Figure2_PC_Evolution_Multipanel.png'))
display(Image('/kaggle/working/PCA_System - 2/Figure3_Folding_Metrics_Dashboard.png'))
display(Image('/kaggle/working/PCA_System - 2/Figure4_Variance_Analysis.png'))



DNA FOLD AND CONTACT MAP PCA ANALYSIS PIPELINE
Input directory: /kaggle/input/cross-validation-dataset-2/give_partham_data/contactmap/system1/npy_data
Output directory: /kaggle/working/PCA_System - 2

[Step 1/6] Loading contact maps...
Loading contact maps from: /kaggle/input/cross-validation-dataset-2/give_partham_data/contactmap/system1/npy_data
Found 200 npz files
  Loaded 10/200 files...
  Loaded 20/200 files...
  Loaded 30/200 files...
  Loaded 40/200 files...
  Loaded 50/200 files...
  Loaded 60/200 files...
  Loaded 70/200 files...
  Loaded 80/200 files...
  Loaded 90/200 files...
  Loaded 100/200 files...
  Loaded 110/200 files...
  Loaded 120/200 files...
  Loaded 130/200 files...
  Loaded 140/200 files...
  Loaded 150/200 files...
  Loaded 160/200 files...
  Loaded 170/200 files...
  Loaded 180/200 files...
  Loaded 190/200 files...
  Loaded 200/200 files...

✓ Successfully loaded 200 contact maps
  Contact map shape: (1, 1)
  Matrix dimensions: 1 x 1
  File type: npz
⚠  Val

ValueError: Found array with 0 feature(s) (shape=(200, 0)) while a minimum of 1 is required by StandardScaler.